In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
base_dir = Path(".").resolve()
data_dir = base_dir / "data"

In [3]:
def process_list(row):
    if row is None:
        return []
    else:
        return [int(item) for item in row if isinstance(item, str) and item.isdigit()]

In [4]:
# Distribution Rules

with open(data_dir / "metadata.json") as f:
    metadata = json.load(f)

with open(data_dir / "distribution_rules.json") as f:
    distribution_rules = json.load(
        f, object_hook=lambda obj: {**obj.pop("lvl", {}), **obj}
    )

rules = distribution_rules.get("rules", [])

for rule in rules:
    rrg = rule.get("rrg")
    if rrg is None:
        rule["rrg"] = {}

    for key in list(rule.keys()):
        if key in metadata:
            rule[metadata[key]] = rule.pop(key)

    level_mapping = metadata["level_mapping"]
    for key, value in rule.items():
        if isinstance(value, dict):
            for k in list(value.keys()):
                if k in level_mapping:
                    value[level_mapping[k]] = value.pop(k)

df_rules = pd.json_normalize(rules, sep="_")
df_rules["credential_list"] = df_rules["credential_list"].apply(process_list)
df_rules["credential_count"] = df_rules["credential_list"].apply(len)
df_rules["provider_list"] = df_rules["provider_list"].apply(
    lambda x: x if x is not None else []
)
df_rules["provider_count"] = df_rules["provider_list"].apply(len)
df_rules["hotel_list"] = df_rules["hotel_list"].apply(process_list)
df_rules["hotel_count"] = df_rules["hotel_list"].apply(len)
df_rules = df_rules[
    [
        "id",
        "edit_state",
        "obsolete",
        "name",
        "description",
        "tag",
        "credential_choices",
        "credential_list",
        "credential_count",
        "rate",
        "provider_choices",
        "provider_list",
        "provider_count",
        "hotel_choices",
        "hotel_list",
        "hotel_count",
        "destination_choices",
        "destination_list",
        "refundable",
        "meal_choices",
        "meal_list",
        "market_choices",
        "market_list",
        "dynamic_commission",
        "check_in_choices",
        "check_in_from",
        "check_in_to",
        "booking_date_choices",
        "booking_date_from",
        "booking_date_to",
        "range_choices",
        "range_from",
        "range_to",
        "max_release",
        "days_of_week_choices",
        "days_of_week_list",
        "age",
        "room_choices",
        "room_list",
        "ps_choices",
        "ps_list",
        "num_of_nights_choices",
        "num_of_nights_list",
        "hours_choices",
        "hours_list",
        "updated_by",
        "updated_on",
    ]
]

In [5]:
# Credential Group

# with open(data_dir / "credential_group.json") as f:
#     credential_group = json.load(f)

# df_credential_group = pd.DataFrame(credential_group.get("clientGroupList", []))
# df_credential_group.rename(columns={"clientCodes": "credential_list"}, inplace=True)
# df_credential_group["id"] = df_credential_group["id"].astype(int)
# df_credential_group["credential_list"] = df_credential_group["credential_list"].apply(
#     process_list
# )

In [6]:
# df_rules_credential = df_rules.explode("credential_list")[
#     ["id", "credential_choices", "credential_list"]
# ]

In [7]:
# with pd.option_context("display.max_columns", None):
#     display(df_rules[df_rules["destination_choices"] != 0].head())

In [8]:
df_rules_level_one = df_rules.loc[
    (df_rules["credential_choices"] == 1)
    & (df_rules["provider_choices"] == 1)
    & (df_rules["hotel_choices"] == 1)
    & (df_rules["tag"] == 1)
]

In [9]:
df_rules_level_one = df_rules_level_one.sort_values(
    by="credential_count", ascending=False
)

In [10]:
df_exploded = (
    df_rules_level_one.explode("credential_list")
    .explode("provider_list")
    .explode("hotel_list")
)

In [11]:
df_exploded["combination_zip"] = list(
    zip(
        df_exploded["credential_list"],
        df_exploded["provider_list"],
        df_exploded["hotel_list"],
    )
)

In [12]:
df_exploded["is_duplicate"] = df_exploded.duplicated(
    subset="combination_zip", keep="first"
)

In [13]:
duplicate_ids = (
    df_exploded[df_exploded["is_duplicate"] | ~df_exploded["is_duplicate"]]
    .groupby("combination_zip")["id"]
    .apply(list)
    .reset_index(name="duplicate_ids")
)

In [14]:
df_exploded = df_exploded.merge(duplicate_ids, on="combination_zip", how="left")

In [15]:
duplicates = df_exploded[df_exploded["is_duplicate"]]
duplicates.insert(0, "duplicate_ids", duplicates.pop("duplicate_ids"))

In [16]:
df_exploded.drop_duplicates(subset=["combination_zip"], keep="first", inplace=True)

In [20]:
df_exploded = (
    df_exploded.groupby("id")
    .agg(
        {
            "credential_list": lambda x: list(set(x)),
            "provider_list": lambda x: list(set(x)),
            "hotel_list": lambda x: list(set(x)),
            "duplicate_ids": "first",
        }
    )
    .reset_index()
)

In [21]:
duplicates = (
    duplicates.groupby("id")
    .agg(
        {
            "credential_list": lambda x: list(set(x)),
            "provider_list": lambda x: list(set(x)),
            "hotel_list": lambda x: list(set(x)),
            "duplicate_ids": "first",
        }
    )
    .reset_index()
)

In [28]:
change_data = []

for index, row in df_rules_level_one.iterrows():
    id_ = row["id"]

    # Original lists for this ID
    original_credential_list = row["credential_list"]
    original_provider_list = row["provider_list"]
    original_hotel_list = row["hotel_list"]

    # Get duplicate IDs for the current ID
    duplicate_ids = duplicates.loc[duplicates["id"] == id_, "duplicate_ids"].values
    duplicate_ids = duplicate_ids[0] if duplicate_ids.size > 0 else []

    # Updated lists after deletion (removing duplicates)
    updated_credential_list = (
        df_exploded.loc[df_exploded["id"] == id_, "credential_list"]
        .apply(lambda x: x if isinstance(x, list) else [])
        .explode()
        .unique()
        .tolist()
    )
    updated_provider_list = (
        df_exploded.loc[df_exploded["id"] == id_, "provider_list"]
        .apply(lambda x: x if isinstance(x, list) else [])
        .explode()
        .unique()
        .tolist()
    )
    updated_hotel_list = (
        df_exploded.loc[df_exploded["id"] == id_, "hotel_list"]
        .apply(lambda x: x if isinstance(x, list) else [])
        .explode()
        .unique()
        .tolist()
    )

    # Find deleted items
    deleted_credential_list = list(
        set(original_credential_list) - set(updated_credential_list)
    )
    deleted_provider_list = list(
        set(original_provider_list) - set(updated_provider_list)
    )
    deleted_hotel_list = list(set(original_hotel_list) - set(updated_hotel_list))

    # Append the results to the change data list
    change_data.append(
        {
            "id": id_,
            "duplicate_ids": duplicate_ids,
            "credential_list": original_credential_list,
            "updated_credential_list": updated_credential_list,
            "deleted_credential_list": deleted_credential_list,
            "provider_list": original_provider_list,
            "updated_provider_list": updated_provider_list,
            "deleted_provider_list": deleted_provider_list,
            "hotel_list": original_hotel_list,
            "updated_hotel_list": updated_hotel_list,
            "deleted_hotel_list": deleted_hotel_list,
            "edit_state":row["edit_state"],
            "obsolete":row["obsolete"],
            "name":row["name"],
            "description":row["description"],
            "tag":row["tag"],
            "credential_choices":row["credential_choices"],
            "credential_count":row["credential_count"],
            "rate":row["rate"],
            "provider_choices":row["provider_choices"],
            "provider_count":row["provider_count"],
            "hotel_choices":row["hotel_choices"],
            "hotel_count":row["hotel_count"],
            "destination_choices":row["destination_choices"],
            "destination_list":row["destination_list"],
            "refundable":row["refundable"],
            "meal_choices":row["meal_choices"],
            "meal_list":row["meal_list"],
            "market_choices":row["market_choices"],
            "market_list":row["market_list"],
            "dynamic_commission":row["dynamic_commission"],
            "check_in_choices":row["check_in_choices"],
            "check_in_from":row["check_in_from"],
            "check_in_to":row["check_in_to"],
            "booking_date_choices":row["booking_date_choices"],
            "booking_date_from":row["booking_date_from"],
            "booking_date_to":row["booking_date_to"],
            "range_choices":row["range_choices"],
            "range_from":row["range_from"],
            "range_to":row["range_to"],
            "max_release":row["max_release"],
            "days_of_week_choices":row["days_of_week_choices"],
            "days_of_week_list":row["days_of_week_list"],
            "age":row["age"],
            "room_choices":row["room_choices"],
            "room_list":row["room_list"],
            "ps_choices":row["ps_choices"],
            "ps_list":row["ps_list"],
            "num_of_nights_choices":row["num_of_nights_choices"],
            "num_of_nights_list":row["num_of_nights_list"],
            "hours_choices":row["hours_choices"],
            "hours_list":row["hours_list"],
            "updated_by":row["updated_by"],
            "updated_on":row["updated_on"],

        }
    )

In [29]:
change_tracker = pd.DataFrame(change_data)
change_tracker.to_csv("output.csv", index=False)